# 20 – Census Boundaries Cleaning (Tracts / Block Groups)

This notebook prepares **census geometries** for use with ACS and LODES data.

**Goals:**
- Load raw census shapefiles (tracts, block groups, etc.).
- Inspect CRS and standardize to the project CRS.
- Keep only the study area (e.g., Maricopa County / City of Phoenix).
- Keep only essential ID fields (e.g., `GEOID`) and geometry.
- Save cleaned boundary layers for joining to ACS and LODES later.

In [20]:
from pathlib import Path
import geopandas as gpd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CENSUS_DIR = DATA_DIR / "census"
BOUNDARIES_DIR = DATA_DIR / "census/boundaries"

CENSUS_DIR, BOUNDARIES_DIR

(WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/census'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/census/boundaries'))

In [21]:
raw_tracts_path = BOUNDARIES_DIR / "US_tract_2023.shp"

tracts_clean_path = CENSUS_DIR / "processed/tracts_clean.gpkg"

raw_tracts_path, tracts_clean_path

(WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/census/boundaries/US_tract_2023.shp'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/census/processed/tracts_clean.gpkg'))

In [22]:
tracts = gpd.read_file(raw_tracts_path)

print("Tracts:", len(tracts), " | CRS:", tracts.crs)

tracts.head()

Tracts: 85074  | CRS: PROJCS["USA_Contiguous_Albers_Equal_Area_Conic",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",37.5],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","102003"]]


,GISJOIN,STATEFP,COUNTYFP,TRACTCE,GEOID,GEOIDFQ,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,Shape_Leng,Shape_Area,ORIG_FID,geometry
0,G0100010020100,01,001,020100,01001020100,1400000US01001020100,201,Census Tract 201,G5020,S,9825303.0,28435.0,+32.4819731,-086.4915648,16217.763443,9.853735e+06,28595,"POLYGON ((888437.995 -515917.537, 888435.804 -..."
1,G0100010020200,01,001,020200,01001020200,1400000US01001020200,202,Census Tract 202,G5020,S,3320818.0,5669.0,+32.4757580,-086.4724678,9824.372415,3.326483e+06,28593,"POLYGON ((889844.072 -519142.061, 889844.876 -..."
2,G0100010020300,01,001,020300,01001020300,1400000US01001020300,203,Census Tract 203,G5020,S,5349271.0,9054.0,+32.4740243,-086.4597033,10519.641206,5.358327e+06,28734,"POLYGON ((891383.841 -518871.184, 891367.251 -..."
3,G0100010020400,01,001,020400,01001020400,1400000US01001020400,204,Census Tract 204,G5020,S,6384282.0,8408.0,+32.4710304,-086.4448353,12521.196228,6.392683e+06,28596,"POLYGON ((892527.268 -516528.67, 892531.715 -5..."
4,G0100010020501,01,001,020501,01001020501,1400000US01001020501,205.01,Census Tract 205.01,G5020,S,6203654.0,0.0,+32.4478607,-086.4225578,11422.446991,6.203654e+06,28164,"POLYGON ((895018.44 -518564.833, 895058.489 -5..."


In [23]:
# fill with actual CRS from 00_project_setup
PROJECT_CRS = "EPSG:2868"
PROJECT_CRS

'EPSG:2868'

In [24]:
def ensure_project_crs(gdf, name="layer"):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:2868")
        print(f"{name}: CRS is None – set manually before reprojecting.")
        return gdf
    
    if PROJECT_CRS is not None and gdf.crs != PROJECT_CRS:
        print(f"{name}: Reprojecting from {gdf.crs} to {PROJECT_CRS}")
        return gdf.to_crs(PROJECT_CRS)
    else:
        print(f"{name}: No reprojection needed.")
        return gdf

tracts = ensure_project_crs(tracts, name="tracts")

tracts: Reprojecting from PROJCS["USA_Contiguous_Albers_Equal_Area_Conic",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",37.5],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","102003"]] to EPSG:2868


In [25]:
# Since the file is national, filter to Maricopa County

STATE_FIELD = "STATEFP"
STATE_CODE = "04"  # Arizona FIPS code
COUNTY_FIELD = "COUNTYFP"
COUNTY_CODE = "013"  # Maricopa County FIPS code

tracts = tracts[(tracts[STATE_FIELD] == STATE_CODE) & (tracts[COUNTY_FIELD] == COUNTY_CODE)].copy()
len(tracts)

1009

In [ ]:
print(tracts.columns)

Index(['GISJOIN', 'STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOID', 'GEOIDFQ', 'NAME',
       'NAMELSAD', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT',
       'INTPTLON', 'Shape_Leng', 'Shape_Area', 'ORIG_FID', 'geometry'],
      dtype='object')


In [27]:
TRACT_GEOID_FIELD = "GISJOIN"

tracts_clean = tracts[[TRACT_GEOID_FIELD, "geometry"]].copy()

tracts_clean.head()

,GISJOIN,geometry
1747,G0400130010102,"POLYGON ((827801.708 1091626.91, 827865.552 10..."
1748,G0400130010103,"POLYGON ((765755.318 1002785.693, 765592.367 1..."
1749,G0400130010104,"POLYGON ((765755.318 1002785.693, 765757.331 1..."
1750,G0400130030401,"POLYGON ((707726.563 1037891.22, 707726.326 10..."
1751,G0400130030402,"POLYGON ((707726.563 1037891.22, 707707.434 10..."


In [31]:
tracts_clean.to_file(
    tracts_clean_path,
    driver="GPKG")

print("Saved cleaned tracts:", tracts_clean_path)

Saved cleaned tracts: c:\Users\arjav\DevSource\toc-performance-dashboard\data\census\processed\tracts_clean.gpkg
